# Phase 3: Build a Strong Retrieval Pipeline

## Step 7: Hybrid Search

### Learning

- Hybrid retrieval
- Candidate generation
- Score normalization
- Rank fusion
- Reciprocal Rank Fusion (RRF)
- Weighted retrieval
- Deduplication
- Retrieval recall

---

## Key Takeaways

- Lexical (BM25/Elasticsearch) and semantic (embeddings/Chroma) retrieval solve different problems and fail in different ways.
- Raw BM25 scores and cosine-similarity scores live on incompatible scales — adding them directly is misleading.
- **Reciprocal Rank Fusion (RRF)** combines *rank positions* instead of raw scores, which sidesteps the scale-mismatch problem.
- A chunk may be returned by only one retriever — the merge step must handle that gracefully.
- Hybrid search only improves the *candidate set*; it doesn't guarantee the best chunk is ranked first (that's what reranking, Step 8, is for).

---

## To do (mirrors the Roadmap 1:1)

1. Create a common retrieval result format
2. Run both retrievers for every query
3. Merge results by stable ID
4. Implement Reciprocal Rank Fusion
5. Avoid combining raw scores directly (normalize as a *separate* experiment)
6. Deduplicate similar results
7. Add retrieval-mode controls (vector / BM25 / hybrid-RRF)
8. Create a test query set
9. Experiment with weighted fusion
10. Send fused results to the model, with citations

## 0. Environment Setup — connect to both retrieval backends

Same connection + indexing pattern as Steps 4–6: Elasticsearch for lexical search,
Chroma for vector search. Nothing new conceptually here — just getting both
backends populated with the same document set so Step 7 can query them side by side.

In [1]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")

try:
    response = es.info()
    print(response)

except Exception as e:
    print("TYPE:", type(e).__name__)
    print("ERROR:")
    print(e)

    if hasattr(e, "body"):
        print("\nBODY:")
        print(e.body)

import sys
from pathlib import Path
PROJECT_ROOT = Path(".." ).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

## helper method to reload specific file
import importlib
import config
importlib.reload(config)

TYPE: ConnectionError
ERROR:
Connection error caused by: ConnectionError(Connection error caused by: NewConnectionError(HTTPConnection(host='localhost', port=9200): Failed to establish a new connection: [Errno 61] Connection refused))


<module 'config' from '/Users/hirakhan/Developer/AI-ML/rag-chatbot/config.py'>

In [5]:
import time
import numpy as np
import chromadb
from openai import OpenAI
from config import OPENAI_API_KEY, MODEL_NAME, EMBEDDING_MODEL

client = OpenAI(api_key=OPENAI_API_KEY)
client_chroma = chromadb.HttpClient(
    host="localhost",
    port=8000,
)

INDEX_NAME = "rag_documents_hybrid"
COLLECTION_NAME = "john_doe_profile_hybrid"

ValueError: Could not connect to a Chroma server. Are you sure it is running?

In [10]:
# Load document & split into paragraphs
document_path = PROJECT_ROOT / "data" / "profile.txt"
document_text = document_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in document_text.split("\n\n") if p.strip()]
documents = [{"id": i, "text": p} for i, p in enumerate(paragraphs)]
print(f"Loaded {len(documents)} chunks.")

Loaded 22 chunks.


In [12]:
# Index into Elasticsearch (lexical side) — same pattern as Step 6
from elasticsearch.helpers import bulk

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

mapping = {
    "mappings": {
        "properties": {
            "chunk_id": {"type": "keyword"},
            "text": {"type": "text"}
        }
    }
}
es.indices.create(index=INDEX_NAME, body=mapping)

actions = [
    {
        "_index": INDEX_NAME,
        "_id": str(doc["id"]),
        "_source": {"chunk_id": str(doc["id"]), "text": doc["text"]}
    }
    for doc in documents
]
success, failed = bulk(es, actions)
print("Successfully indexed into Elasticsearch:", success, "| Failed:", failed)

Successfully indexed into Elasticsearch: 22 | Failed: []


In [13]:
# Index into Chroma (semantic side) — same pattern as Steps 4/5
try:
    client_chroma.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = client_chroma.get_or_create_collection(name=COLLECTION_NAME)

embedded_documents = []
for doc in documents:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=doc["text"])
    embedded_documents.append({
        "id": doc["id"],
        "text": doc["text"],
        "embedding": response.data[0].embedding
    })

collection.add(
    ids=[str(doc["id"]) for doc in embedded_documents],
    documents=[doc["text"] for doc in embedded_documents],
    embeddings=[doc["embedding"] for doc in embedded_documents],
)

print(f"Indexed {collection.count()} documents into Chroma.")

Indexed 22 documents into Chroma.


## 1. Create a common retrieval result format

> Make Chroma and Elasticsearch return results in the same structure. This makes it
> easier to combine results from different systems.

Every retriever — regardless of backend — should return a list of dicts shaped like:

```python
{
    "chunk_id": "chunk-001",
    "text": "...",
    "source": "profile.txt",
    "rank": 1,
    "score": 0.87,
    "retriever": "vector"
}
```

`rank` is 1-based position within that retriever's own result list. `score` stays in
the retriever's *native* scale for now (BM25 relevance score, or cosine similarity) —
we deliberately do NOT normalize it here (see Section 5).

In [15]:
def make_result(chunk_id, text, source, rank, score, retriever):
    """Build one result dict in the common format shared by every retriever."""
    return {
        "chunk_id": str(chunk_id),
        "text": text,
        "source": source,
        "rank": rank,
        "score": score,
        "retriever": retriever,
    }

## 2. Run both retrievers for every query

> For each user query: (1) generate the query embedding, (2) retrieve the top ten
> vector results, (3) send the original query to Elasticsearch, (4) retrieve the top
> ten lexical results. Record the latency of each retrieval method separately.

Both functions below return results already shaped in the common format from
Section 1, and both time themselves independently so you can compare latency.

In [16]:
def get_embedding(text):
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=text)
    return response.data[0].embedding


def vector_search(query_text, top_k=10):
    """Semantic retrieval via Chroma. Returns (results, latency_ms)."""
    t0 = time.perf_counter()

    query_embedding = get_embedding(query_text)
    raw = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["documents", "distances"]
    )

    ids = raw["ids"][0]
    texts = raw["documents"][0]
    distances = raw["distances"][0]

    results = [
        make_result(
            chunk_id=doc_id,
            text=text,
            source="profile.txt",
            rank=i + 1,
            score=1 / (1 + distance),  # distance -> similarity, native to this retriever
            retriever="vector",
        )
        for i, (doc_id, text, distance) in enumerate(zip(ids, texts, distances))
    ]

    latency_ms = (time.perf_counter() - t0) * 1000
    return results, latency_ms


def lexical_search(query_text, top_k=10):
    """Lexical (BM25) retrieval via Elasticsearch. Returns (results, latency_ms)."""
    t0 = time.perf_counter()

    query = {
        "size": top_k,
        "query": {"match": {"text": query_text}}
    }
    raw = es.search(index=INDEX_NAME, body=query)

    results = [
        make_result(
            chunk_id=hit["_source"]["chunk_id"],
            text=hit["_source"]["text"],
            source="profile.txt",
            rank=i + 1,
            score=hit["_score"],
            retriever="bm25",
        )
        for i, hit in enumerate(raw["hits"]["hits"])
    ]

    latency_ms = (time.perf_counter() - t0) * 1000
    return results, latency_ms


def run_both_retrievers(query_text, top_k=10):
    """Run vector + lexical retrieval for one query and report latency separately."""
    vector_results, vector_latency_ms = vector_search(query_text, top_k=top_k)
    lexical_results, lexical_latency_ms = lexical_search(query_text, top_k=top_k)

    print(f"Query: '{query_text}'")
    print(f"  vector retrieval:  {len(vector_results):>2} results in {vector_latency_ms:6.2f} ms")
    print(f"  lexical retrieval: {len(lexical_results):>2} results in {lexical_latency_ms:6.2f} ms")

    return vector_results, lexical_results


_ = run_both_retrievers("John innovation")

Query: 'John innovation'
  vector retrieval:  10 results in 1270.81 ms
  lexical retrieval: 10 results in  41.35 ms


## 3. Merge results by stable ID

> Use `chunk_id` to identify the same chunk across both systems. Create a merged
> result object containing vector rank, vector score, BM25 rank, BM25 score, source
> metadata, and original text. Handle chunks returned by only one retriever.

This is the first piece for you to implement. Build a dict keyed by `chunk_id` where
each entry tracks what each retriever saw for that chunk — defaulting the missing
side to `None` rather than skipping it.

In [19]:
def merge_by_chunk_id(vector_results, lexical_results):
    """Merge vector + lexical result lists into one dict keyed by chunk_id.

    Returns:
        {
            chunk_id: {
                "chunk_id": ...,
                "text": ...,
                "source": ...,
                "vector_rank": int | None,
                "vector_score": float | None,
                "bm25_rank": int | None,
                "bm25_score": float | None,
            },
            ...
        }
    """
    merged = {}

    # TODO: iterate vector_results.
    #   For each result, get-or-create merged[chunk_id] with the fields above
    #   (text/source only need to be set once), then fill in vector_rank/vector_score.

    # TODO: iterate lexical_results the same way, filling in bm25_rank/bm25_score.
    #   A chunk_id may already exist from the vector pass — don't overwrite text/source,
    #   just add the bm25_* fields.


    # -----------------------------
    # 1. Add vector search results
    # -----------------------------
    for rank, result in enumerate(vector_results, start=1):
        chunk_id = result["chunk_id"]

        # Create the entry if this chunk hasn't been seen yet
        if chunk_id not in merged:
            merged[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result.get("text"),
                "source": result.get("source"),
                "vector_rank": None,
                "vector_score": None,
                "bm25_rank": None,
                "bm25_score": None,
            }

        # Fill vector-specific information
        merged[chunk_id]["vector_rank"] = rank
        merged[chunk_id]["vector_score"] = result.get("score")


    # -----------------------------
    # 2. Add BM25 / lexical results
    # -----------------------------
    for rank, result in enumerate(lexical_results, start=1):
        chunk_id = result["chunk_id"]

        # Create the entry if this chunk only exists
        # in lexical results
        if chunk_id not in merged:
            merged[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result.get("text"),
                "source": result.get("source"),
                "vector_rank": None,
                "vector_score": None,
                "bm25_rank": None,
                "bm25_score": None,
            }

        # Fill BM25-specific information
        merged[chunk_id]["bm25_rank"] = rank
        merged[chunk_id]["bm25_score"] = result.get("score")


    return merged
    # raise NotImplementedError("Merge vector_results and lexical_results by chunk_id")



# Try it:
vector_results, lexical_results = run_both_retrievers("John innovation")
merged = merge_by_chunk_id(vector_results, lexical_results)
for chunk_id, entry in merged.items():
    print(
        chunk_id,
        "vector_rank =", entry["vector_rank"],
        "bm25_rank =", entry["bm25_rank"]
    )

Query: 'John innovation'
  vector retrieval:  10 results in 2541.24 ms
  lexical retrieval: 10 results in  26.74 ms
8 vector_rank = 1 bm25_rank = None
18 vector_rank = 2 bm25_rank = 3
15 vector_rank = 3 bm25_rank = 6
11 vector_rank = 4 bm25_rank = None
16 vector_rank = 5 bm25_rank = 9
9 vector_rank = 6 bm25_rank = None
21 vector_rank = 7 bm25_rank = 2
19 vector_rank = 8 bm25_rank = None
0 vector_rank = 9 bm25_rank = 1
2 vector_rank = 10 bm25_rank = 4
1 vector_rank = None bm25_rank = 5
20 vector_rank = None bm25_rank = 7
7 vector_rank = None bm25_rank = 8
3 vector_rank = None bm25_rank = 10


## 4. Implement Reciprocal Rank Fusion

> `RRF score = Σ 1 / (k + rank)`. For every result: (1) calculate its contribution
> from vector rank, (2) calculate its contribution from BM25 rank, (3) add both
> contributions, (4) sort by the final RRF score. Expose `k` as a configurable
> parameter.

RRF combines **rank positions**, not raw scores — that's what makes it robust to the
scale mismatch between cosine similarity and BM25 (see Section 5). A chunk missing
from one retriever simply contributes 0 for that side.

In [20]:
def reciprocal_rank_fusion(merged, k=60):
    """Compute RRF scores for a merged result dict (from Section 3).

    Args:
        merged: output of merge_by_chunk_id().
        k: RRF constant. 60 is the commonly used default.

    Returns:
        List of merged entries with an added "rrf_score" field,
        sorted descending by rrf_score.
    """
    
    results = []

    for chunk_id, entry in merged.items():

        # Vector contribution
        if entry["vector_rank"] is not None:
            vector_contribution = 1 / (k + entry["vector_rank"])
        else:
            vector_contribution = 0


        # BM25 contribution
        if entry["bm25_rank"] is not None:
            bm25_contribution = 1 / (k + entry["bm25_rank"])
        else:
            bm25_contribution = 0


        # Combined RRF score
        rrf_score = vector_contribution + bm25_contribution

        # Store the score
        entry["rrf_score"] = rrf_score

        # Add result to list
        results.append(entry)


    # Sort highest RRF score first
    results.sort(
        key=lambda x: x["rrf_score"],
        reverse=True
    )

    return results

In [21]:
rrf_results = reciprocal_rank_fusion(merged)

for result in rrf_results:
    print(
        result["chunk_id"],
        "vector_rank =", result["vector_rank"],
        "bm25_rank =", result["bm25_rank"],
        "rrf_score =", result["rrf_score"]
    )

18 vector_rank = 2 bm25_rank = 3 rrf_score = 0.03200204813108039
21 vector_rank = 7 bm25_rank = 2 rrf_score = 0.031054405392392875
15 vector_rank = 3 bm25_rank = 6 rrf_score = 0.031024531024531024
0 vector_rank = 9 bm25_rank = 1 rrf_score = 0.030886196246139225
2 vector_rank = 10 bm25_rank = 4 rrf_score = 0.029910714285714284
16 vector_rank = 5 bm25_rank = 9 rrf_score = 0.029877369007803793
8 vector_rank = 1 bm25_rank = None rrf_score = 0.01639344262295082
11 vector_rank = 4 bm25_rank = None rrf_score = 0.015625
1 vector_rank = None bm25_rank = 5 rrf_score = 0.015384615384615385
9 vector_rank = 6 bm25_rank = None rrf_score = 0.015151515151515152
20 vector_rank = None bm25_rank = 7 rrf_score = 0.014925373134328358
19 vector_rank = 8 bm25_rank = None rrf_score = 0.014705882352941176
7 vector_rank = None bm25_rank = 8 rrf_score = 0.014705882352941176
3 vector_rank = None bm25_rank = 10 rrf_score = 0.014285714285714285


## 5. Avoid combining raw scores directly

> Keep cosine similarity and BM25 scores separate. Their ranges and distributions
> are different, so directly adding them can produce misleading results. As a
> separate experiment, normalize scores and compare the results with rank fusion.

This section is intentionally **not** used inside `reciprocal_rank_fusion` above —
RRF never touches raw scores. `normalize_scores` exists purely so you can run the
alternative experiment: min-max normalize each retriever's scores to `[0, 1]`, then
combine with a weighted sum, and compare that ranking against RRF's ranking.

In [22]:
def normalize_scores(results):
    """Min-max normalize the 'score' field of a list of result dicts to [0, 1].

    Adds a 'normalized_score' field to each dict in place and returns the list.
    If all scores are identical (or the list is empty), normalized_score = 1.0
    to avoid divide-by-zero.
    """
    if not results:
        return results

    scores = [r["score"] for r in results]
    min_score, max_score = min(scores), max(scores)

    if max_score == min_score:
        for r in results:
            r["normalized_score"] = 1.0
        return results

    for r in results:
        r["normalized_score"] = (r["score"] - min_score) / (max_score - min_score)

    return results


# Experiment scaffold — compare normalized-score fusion vs RRF on the same query:
# vector_results, lexical_results = run_both_retrievers("John innovation")
# normalize_scores(vector_results)
# normalize_scores(lexical_results)
# TODO: build a normalized-score-weighted ranking the same way merge_by_chunk_id
#   does, but summing normalized_score instead of using rank, then compare the
#   resulting order against reciprocal_rank_fusion()'s order for the same query.

In [23]:
def normalized_score_fusion(
    vector_results,
    lexical_results,
    vector_weight=0.5,
    lexical_weight=0.5
):
    """
    Combine vector and lexical results using normalized scores.

    Args:
        vector_results: Results from vector search.
        lexical_results: Results from BM25 / Elasticsearch.
        vector_weight: Weight given to vector scores.
        lexical_weight: Weight given to lexical scores.

    Returns:
        Combined results sorted by fused_score descending.
    """

    # Normalize scores first
    normalize_scores(vector_results)
    normalize_scores(lexical_results)

    merged = {}

    # Add vector results
    for result in vector_results:
        chunk_id = result["chunk_id"]

        if chunk_id not in merged:
            merged[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result["text"],
                "source": result["source"],
                "vector_score": None,
                "vector_normalized_score": 0.0,
                "bm25_score": None,
                "bm25_normalized_score": 0.0,
            }

        merged[chunk_id]["vector_score"] = result["score"]
        merged[chunk_id]["vector_normalized_score"] = (
            result["normalized_score"]
        )

    # Add lexical/BM25 results
    for result in lexical_results:
        chunk_id = result["chunk_id"]

        if chunk_id not in merged:
            merged[chunk_id] = {
                "chunk_id": chunk_id,
                "text": result["text"],
                "source": result["source"],
                "vector_score": None,
                "vector_normalized_score": 0.0,
                "bm25_score": None,
                "bm25_normalized_score": 0.0,
            }

        merged[chunk_id]["bm25_score"] = result["score"]
        merged[chunk_id]["bm25_normalized_score"] = (
            result["normalized_score"]
        )

    # Calculate weighted combined score
    results = []

    for entry in merged.values():

        fused_score = (
            vector_weight * entry["vector_normalized_score"]
            + lexical_weight * entry["bm25_normalized_score"]
        )

        entry["fused_score"] = fused_score

        results.append(entry)

    # Highest fused score first
    results.sort(
        key=lambda x: x["fused_score"],
        reverse=True
    )

    return results

In [24]:
vector_results, lexical_results = run_both_retrievers(
    "John innovation"
)

fused_results = normalized_score_fusion(
    vector_results,
    lexical_results
)

for result in fused_results:
    print(
        result["chunk_id"],
        "vector =", result["vector_normalized_score"],
        "bm25 =", result["bm25_normalized_score"],
        "fused =", result["fused_score"]
    )

Query: 'John innovation'
  vector retrieval:  10 results in 1457.69 ms
  lexical retrieval: 10 results in  39.90 ms
0 vector = 0.07480339779709677 bm25 = 1.0 fused = 0.5374016988985484
8 vector = 1.0 bm25 = 0.0 fused = 0.5
18 vector = 0.987277927828145 bm25 = 0.006566859721310957 fused = 0.496922393774728
21 vector = 0.16756413335828382 bm25 = 0.7297908990051933 fused = 0.44867751618173857
15 vector = 0.6553164075269117 bm25 = 0.0011054138767029463 fused = 0.3282109107018073
11 vector = 0.5601193057845189 bm25 = 0.0 fused = 0.2800596528922594
16 vector = 0.4912469554417015 bm25 = 0.0003563863437160748 fused = 0.2458016708927088
9 vector = 0.36940883095367527 bm25 = 0.0 fused = 0.18470441547683764
19 vector = 0.16558880920391278 bm25 = 0.0 fused = 0.08279440460195639
2 vector = 0.0 bm25 = 0.005425716161382819 fused = 0.0027128580806914096
1 vector = 0.0 bm25 = 0.003876717241974229 fused = 0.0019383586209871144
20 vector = 0.0 bm25 = 0.0011054138767029463 fused = 0.0005527069383514732
7 

In [25]:
# RRF
merged = merge_by_chunk_id(
    vector_results,
    lexical_results
)

rrf_results = reciprocal_rank_fusion(merged)

print("RRF ranking:")
for result in rrf_results:
    print(
        result["chunk_id"],
        result["rrf_score"]
    )

print("\nNormalized-score fusion ranking:")

for result in fused_results:
    print(
        result["chunk_id"],
        result["fused_score"]
    )

RRF ranking:
18 0.03200204813108039
21 0.031054405392392875
15 0.031024531024531024
0 0.030886196246139225
2 0.029910714285714284
16 0.029877369007803793
8 0.01639344262295082
11 0.015625
1 0.015384615384615385
9 0.015151515151515152
20 0.014925373134328358
19 0.014705882352941176
7 0.014705882352941176
3 0.014285714285714285

Normalized-score fusion ranking:
0 0.5374016988985484
8 0.5
18 0.496922393774728
21 0.44867751618173857
15 0.3282109107018073
11 0.2800596528922594
16 0.2458016708927088
9 0.18470441547683764
19 0.08279440460195639
2 0.0027128580806914096
1 0.0019383586209871144
20 0.0005527069383514732
7 0.0003623286020868659
3 0.0


## 6. Deduplicate similar results

> Two chunks may contain almost identical content because of chunk overlap,
> duplicate source documents, reindexed documents, or different versions. Start by
> deduplicating exact chunk IDs. Later, add text-hash or similarity-based
> deduplication.

The merge step in Section 3 already deduplicates exact `chunk_id`s (since it's a
dict keyed by `chunk_id`). What's left here is **content-level** deduplication —
different IDs that happen to hold near-identical text.

In [26]:
import hashlib

def content_hash(text):
    """Stable hash of normalized chunk text, for exact-duplicate detection."""
    normalized = " ".join(text.lower().split())
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def deduplicate_by_content_hash(fused_results):
    """Given RRF-sorted results, drop later duplicates that share a content hash
    with an earlier (higher-ranked) result, keeping the first (best-ranked) copy.
    """
    seen_hashes = set()
    deduped = []

    for result in fused_results:
        h = content_hash(result["text"])
        if h in seen_hashes:
            continue
        seen_hashes.add(h)
        deduped.append(result)

    return deduped


# TODO (optional, later): replace exact-hash matching with similarity-based
# dedup — e.g. compare embeddings of near-neighbor chunks and drop any pair
# above a cosine-similarity threshold (~0.97+).

## 7. Add retrieval-mode controls

> Allow the learner to choose: vector only, BM25 only, hybrid with RRF. Display the
> results side by side.

In [27]:
RETRIEVAL_MODE_VECTOR = "vector"
RETRIEVAL_MODE_BM25 = "bm25"
RETRIEVAL_MODE_HYBRID = "hybrid"


def hybrid_search(query_text, mode=RETRIEVAL_MODE_HYBRID, top_k=5, candidate_k=10, rrf_k=60):
    """Single entry point supporting all three retrieval modes.

    vector / bm25 -> just that retriever's top_k, in the common format.
    hybrid        -> run both, merge (Section 3), fuse with RRF (Section 4),
                      dedupe (Section 6), return top_k.
    """
    if mode == RETRIEVAL_MODE_VECTOR:
        results, _ = vector_search(query_text, top_k=top_k)
        return results

    elif mode == RETRIEVAL_MODE_BM25:
        results, _ = lexical_search(query_text, top_k=top_k)
        return results

    elif mode == RETRIEVAL_MODE_HYBRID:
        vector_results, lexical_results = run_both_retrievers(query_text, top_k=candidate_k)
        merged = merge_by_chunk_id(vector_results, lexical_results)
        fused = reciprocal_rank_fusion(merged, k=rrf_k)
        deduped = deduplicate_by_content_hash(fused)
        return deduped[:top_k]

    else:
        raise ValueError(f"Unknown retrieval mode: {mode}")


def compare_modes_side_by_side(query_text, top_k=5):
    """Run all three modes on the same query and print results side by side."""
    for mode in (RETRIEVAL_MODE_VECTOR, RETRIEVAL_MODE_BM25, RETRIEVAL_MODE_HYBRID):
        print(f"\n=== mode: {mode} ===")
        results = hybrid_search(query_text, mode=mode, top_k=top_k)
        for r in results:
            score_field = "rrf_score" if mode == RETRIEVAL_MODE_HYBRID else "score"
            print(f"  chunk={r['chunk_id']:>3}  {score_field}={r[score_field]:.4f}  {r['text'][:70]}...")

## 8. Create a test query set

> Include queries for: exact identifiers, semantic paraphrases, names, acronyms,
> policies, multi-word concepts, very short queries. Record which retriever
> performs better for each query.

In [28]:
test_queries = [
    # category: query
    ("exact_identifier", "EMP-78432"),
    ("semantic_paraphrase", "Where did he go to school?"),
    ("name", "John Doe"),
    ("multi_word_concept", "lifelong learning and mentorship"),
    ("very_short", "innovation"),
]

for category, query in test_queries:
    print(f"\n########## [{category}] \"{query}\" ##########")
    compare_modes_side_by_side(query, top_k=3)

# TODO: after running this, note down (in a markdown cell or comments) which
# retriever won for each category — this observation directly feeds Section 9.


########## [exact_identifier] "EMP-78432" ##########

=== mode: vector ===
  chunk=  8  score=0.3699  Upon completing his degree in 2009, John accepted a position as a juni...
  chunk= 12  score=0.3619  In 2018, John co-founded a fictional startup called BrightPath Technol...
  chunk=  9  score=0.3613  Over the next several years, John gained expertise in full-stack softw...

=== mode: bm25 ===

=== mode: hybrid ===
Query: 'EMP-78432'
  vector retrieval:  10 results in 451.75 ms
  lexical retrieval:  0 results in  11.24 ms
  chunk=  8  rrf_score=0.0164  Upon completing his degree in 2009, John accepted a position as a juni...
  chunk= 12  rrf_score=0.0161  In 2018, John co-founded a fictional startup called BrightPath Technol...
  chunk=  9  rrf_score=0.0159  Over the next several years, John gained expertise in full-stack softw...

########## [semantic_paraphrase] "Where did he go to school?" ##########

=== mode: vector ===
  chunk=  5  score=0.4472  After graduating with honors, Jo

## 9. Experiment with weighted fusion

> Add optional weights: vector weight, lexical weight. For example, give lexical
> retrieval more influence when the query contains numbers, hyphens, capitalized
> codes, or known identifier formats.

This extends RRF with per-retriever weights:
`weighted_rrf_score = vector_weight * (1/(k+vector_rank)) + lexical_weight * (1/(k+bm25_rank))`.

In [29]:
import re

def looks_like_identifier(query_text):
    """Heuristic: does this query look like an exact identifier rather than
    a natural-language question? (numbers, hyphens, capitalized codes, etc.)
    """
    return bool(re.search(r"[0-9]", query_text)) or bool(re.search(r"-", query_text)) \
        or query_text.isupper()


def weighted_rank_fusion(merged, k=60, vector_weight=0.5, lexical_weight=0.5):
    """Same idea as reciprocal_rank_fusion, but each retriever's contribution
    is scaled by a configurable weight before summing.
    """
    results = []

    # TODO: same loop as Section 4's reciprocal_rank_fusion, but multiply
    #   the vector contribution by vector_weight and the bm25 contribution
    #   by lexical_weight before summing into "weighted_score".

    raise NotImplementedError("Implement weighted rank fusion")


def adaptive_hybrid_search(query_text, top_k=5, candidate_k=10, k=60):
    """Automatically favor lexical retrieval for identifier-like queries."""
    if looks_like_identifier(query_text):
        vector_weight, lexical_weight = 0.2, 0.8
    else:
        vector_weight, lexical_weight = 0.5, 0.5

    vector_results, lexical_results = run_both_retrievers(query_text, top_k=candidate_k)
    merged = merge_by_chunk_id(vector_results, lexical_results)
    fused = weighted_rank_fusion(merged, k=k, vector_weight=vector_weight, lexical_weight=lexical_weight)
    deduped = deduplicate_by_content_hash(fused)
    return deduped[:top_k]

## 10. Send fused results to the model

> Select the top five fused chunks. Format them with stable source numbers.
> Ask the model to answer using only the retrieved material.

In [30]:
def format_sources(results):
    """Format fused results with stable [n] source numbers for the prompt."""
    lines = []
    for i, r in enumerate(results, start=1):
        lines.append(f"[{i}] {r['text']}")
    return "\n\n".join(lines)


def answer_with_hybrid_search(user_question, top_k=5):
    results = hybrid_search(user_question, mode=RETRIEVAL_MODE_HYBRID, top_k=top_k)
    sources_block = format_sources(results)

    system_prompt = f"""You answer questions using ONLY the retrieved sources below.
Cite claims using the matching [n] source number.
If the sources do not contain the answer, say so plainly — do not guess.

Retrieved sources:
{sources_block}"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_question},
        ],
    )
    return response.choices[0].message.content, results


answer, sources = answer_with_hybrid_search("What did John study and where?")
print(answer)

Query: 'What did John study and where?'
  vector retrieval:  10 results in 510.06 ms
  lexical retrieval: 10 results in   6.95 ms
John studied Computer Science at the fictional North Valley University, where he pursued a Bachelor of Science degree [2].


## Step 8: Reranking

### Learning

- First-stage retrieval
- Candidate generation
- Second-stage ranking
- Cross-encoders
- Pairwise relevance
- Recall versus precision
- Reranking latency
- Candidate-set size


## 1. Increase first-stage retrieval depth

> Retrieve a broader candidate set: 10 vector results, 10 lexical results, 20 or
> fewer fused candidates. The first stage should prioritize finding potentially
> relevant information rather than perfectly ordering it.

Step 7's `hybrid_search()` already accepts `candidate_k` (per-retriever depth) and
`top_k` (how many fused results to return). For reranking we want a *wider* funnel
than we'd hand straight to the generation model — pull more candidates through the
fusion stage, then let the reranker narrow it back down.

In [31]:
# Widen the first-stage funnel: 10 vector + 10 lexical candidates,
# fused down to at most 20 chunks (instead of Step 7's tighter top_k=5).
RERANK_CANDIDATE_K = 10   # per-retriever depth fed into fusion
RERANK_FUSED_K = 20       # how many fused candidates survive into reranking

def get_rerank_candidates(query_text):
    """Run Step 7's hybrid search wide enough to give the reranker real choices."""
    return hybrid_search(
        query_text,
        mode=RETRIEVAL_MODE_HYBRID,
        top_k=RERANK_FUSED_K,
        candidate_k=RERANK_CANDIDATE_K,
    )


candidates = get_rerank_candidates("What did John study and where?")
print(f"{len(candidates)} candidates ready for reranking")

Query: 'What did John study and where?'
  vector retrieval:  10 results in 1957.04 ms
  lexical retrieval: 10 results in  27.09 ms
14 candidates ready for reranking


## 2. Create a reranking interface

> Define `rerank(query, documents) -> list[dict]`. It should accept the original
> user query and candidate chunks, and return the same chunks with a reranker score
> and reranker rank added.

This is the shared contract every reranker implementation (LLM-based, cross-encoder,
whatever comes next) will follow, so the rest of the pipeline doesn't care which one
is plugged in.

In [32]:
def rerank(query: str, documents: list[dict]) -> list[dict]:
    """Interface every reranker implementation should satisfy.

    Args:
        query: the original user query (not a rewritten/standalone version).
        documents: candidate chunks from Step 7 (common result format).

    Returns:
        The same chunks, each with two added fields:
            "reranker_score": float
            "reranker_rank":  int (1-based, 1 = most relevant)
        sorted descending by reranker_score.
    """
    raise NotImplementedError(
        "Placeholder — Section 3 implements this via an LLM reranker, "
        "call llm_rerank(query, documents) directly for now."
    )

## 3. Build an LLM-based reranker first

> Before using a dedicated reranking model, ask an LLM to score each query-document
> pair. Require structured output: `{"relevance_score": 0, "reason": "..."}`. Use a
> fixed scale such as 0–10. The prompt should ask whether the chunk contains
> information that *helps answer* the query, not whether the chunk is *generally
> related*.

Note the distinction the roadmap draws: "helps answer" vs. "generally related" is
exactly the gap between a reranker and a retriever. A chunk can be topically similar
(high embedding similarity) without containing the actual answer.

In [33]:
import json as _json
from pydantic import BaseModel, Field


class RelevanceScore(BaseModel):
    relevance_score: int = Field(ge=0, le=10)
    reason: str


def score_single_pair(query: str, chunk_text: str) -> RelevanceScore:
    """Ask the LLM to score ONE query-document pair on a 0-10 scale.

    TODO:
      1. Build a prompt that gives the model the query and the chunk text, and
         asks: does this chunk contain information that helps ANSWER the query
         (not just "is it related to the topic")?
      2. Call client.chat.completions.create(...) using MODEL_NAME, requesting
         structured output that matches the RelevanceScore schema (0-10 score
         + a short reason).
      3. Parse and validate the response with RelevanceScore(**parsed_json).
      4. Return the validated RelevanceScore.
    """
    raise NotImplementedError("Implement single-pair LLM scoring")


def llm_rerank(query: str, documents: list[dict]) -> list[dict]:
    """Naive LLM reranker: one model call per candidate (see Section 4 for batching)."""
    scored = []
    for doc in documents:
        result = score_single_pair(query, doc["text"])
        doc = {**doc, "reranker_score": result.relevance_score, "reason": result.reason}
        scored.append(doc)

    scored.sort(key=lambda d: d["reranker_score"], reverse=True)
    for i, doc in enumerate(scored, start=1):
        doc["reranker_rank"] = i

    return scored

## 4. Batch candidates where practical

> Instead of making one model request per candidate, provide several candidates in
> a single request. Require the model to return a score for each stable chunk ID.
> Validate that every chunk received a score, no unknown chunk IDs were created,
> and scores are within the allowed range.

This replaces Section 3's one-call-per-chunk approach with a single call that scores
the whole candidate set — much cheaper and faster for `RERANK_FUSED_K` candidates.

In [34]:
class ChunkScore(BaseModel):
    chunk_id: str
    relevance_score: int = Field(ge=0, le=10)


class BatchRerankResult(BaseModel):
    scores: list[ChunkScore]


def batch_llm_rerank(query: str, documents: list[dict]) -> list[dict]:
    """Score all candidates in a single model call, then validate the response.

    TODO:
      1. Build a prompt listing every candidate as `[chunk_id] text`, and ask the
         model to return a 0-10 relevance score PER chunk_id, matching the
         BatchRerankResult schema.
      2. Call the model, requesting structured output.
      3. Parse into BatchRerankResult.
      4. Validate:
           - every input chunk_id received a score (no missing chunks)
           - no unknown chunk_id appears in the response that wasn't in `documents`
           - every score is within [0, 10] (Pydantic's Field already enforces this,
             but confirm the count matches len(documents))
         If validation fails, decide: retry, fall back to score 0, or raise.
      5. Attach reranker_score to each document, sort descending, assign
         reranker_rank, and return — same output contract as llm_rerank().
    """
    raise NotImplementedError("Implement batched LLM reranking with validation")

## 5. Add a dedicated reranking model

> After understanding the logic, connect a dedicated reranking model or
> cross-encoder. Send query + candidate texts, retrieve relevance scores. Compare
> its latency and ranking with the LLM-based method.

A cross-encoder (e.g. a `sentence-transformers` CrossEncoder model, or a hosted
reranking API like Cohere Rerank) scores query-document pairs directly, without
going through a full chat completion — typically much faster and cheaper per
candidate than an LLM call.

In [35]:
# Optional dependency — install with: pip install sentence-transformers --break-system-packages
#
# from sentence_transformers import CrossEncoder
# cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def cross_encoder_rerank(query: str, documents: list[dict]) -> list[dict]:
    """Rerank using a dedicated cross-encoder model instead of an LLM.

    TODO:
      1. Build (query, chunk_text) pairs for every candidate.
      2. cross_encoder.predict(pairs) -> raw relevance scores (float array).
      3. Attach reranker_score to each document, sort descending, assign
         reranker_rank — same output contract as llm_rerank() / batch_llm_rerank().
      4. Time this function the same way as Section 9 and compare against the
         LLM-based reranker on the same candidate set.
    """
    raise NotImplementedError("Wire up a cross-encoder (or hosted rerank API) and compare")

## 6. Keep only the best chunks

> After reranking, pass only the top three to five chunks to the generation model.
> Record: candidate count before reranking, candidate count after reranking,
> reranking duration, final context token count.

In [36]:
import tiktoken

def finalize_context(query: str, reranked_documents: list[dict], keep_top_n=5):
    """Trim reranked candidates down to the final generation context and report stats."""
    final_chunks = reranked_documents[:keep_top_n]

    encoding = tiktoken.encoding_for_model("gpt-4o")  # any cl100k/o200k-family encoder is fine for counting
    final_context_tokens = sum(len(encoding.encode(c["text"])) for c in final_chunks)

    stats = {
        "candidates_before_rerank": len(reranked_documents),
        "candidates_after_rerank": len(final_chunks),
        "final_context_tokens": final_context_tokens,
    }
    return final_chunks, stats


# candidates = get_rerank_candidates("What did John study and where?")
# reranked = batch_llm_rerank("What did John study and where?", candidates)
# final_chunks, stats = finalize_context("...", reranked, keep_top_n=5)
# print(stats)

## 7. Show before-and-after ranking

> Display a table with: chunk ID, vector rank, BM25 rank, RRF rank, reranker rank,
> reranker score. Identify chunks that moved significantly.

This stitches together every rank Step 7 + Step 8 produced for the same chunk, so
you can see exactly what reranking changed versus first-stage fusion.

In [37]:
def show_before_after(query_text, keep_top_n=5, move_threshold=3):
    """Print a before/after ranking table for one query.

    TODO:
      1. Get first-stage fused candidates AND their pre-rerank order:
           - re-run merge_by_chunk_id + reciprocal_rank_fusion (Step 7) to capture
             vector_rank / bm25_rank / rrf-implied rank per chunk_id
           - or extend get_rerank_candidates() to also return the merged dict
      2. Assign an "rrf_rank" = 1-based position in the RRF-sorted candidate list.
      3. Run batch_llm_rerank() (or cross_encoder_rerank()) on those candidates to
         get reranker_score / reranker_rank.
      4. Print one row per chunk: chunk_id, vector_rank, bm25_rank, rrf_rank,
         reranker_rank, reranker_score.
      5. Flag rows where abs(rrf_rank - reranker_rank) >= move_threshold as
         "moved significantly".
    """
    raise NotImplementedError("Build the combined before/after ranking table")

## 8. Create difficult retrieval examples

> Test cases where: several chunks discuss the same broad topic, only one chunk
> directly answers the question, a high-similarity chunk lacks the answer, an exact
> keyword match is misleading, the answer contains a negation or exception.

In [38]:
difficult_queries = [
    # category: query — fill in / adapt to your profile.txt content
    ("same_broad_topic", "Tell me about John's career."),               # many chunks mention career
    ("one_chunk_has_answer", "What is John's employee ID?"),            # only one chunk has the exact fact
    ("high_similarity_no_answer", "What programming language does John prefer?"),
    ("misleading_keyword_match", "John Smith"),                         # exact-keyword trap if no John Smith exists
    ("negation_or_exception", "When is John NOT available for meetings?"),
]

# TODO: for each (category, query), run get_rerank_candidates() then
# batch_llm_rerank(), and manually inspect whether the top reranked chunk
# actually contains the answer. Note failures — these become Step 9's
# regression/eval fodder later in the roadmap.

## 9. Measure the latency trade-off

> Track: embedding time, vector-search time, Elasticsearch time, fusion time,
> reranking time, generation time, total response time. Determine whether
> reranking should run for every query.

In [40]:
def timed_end_to_end(query_text, keep_top_n=5):
    """Run the full Step 7 + Step 8 pipeline once, timing every stage separately."""
    timings = {}

    t0 = time.perf_counter()
    vector_results, lexical_results = run_both_retrievers(query_text, top_k=RERANK_CANDIDATE_K)
    timings["retrieval_total_ms"] = (time.perf_counter() - t0) * 1000
    # NOTE: vector_search()/lexical_search() already report their own latency
    # individually if you want a finer breakdown than the combined number above.

    t1 = time.perf_counter()
    merged = merge_by_chunk_id(vector_results, lexical_results)
    fused = reciprocal_rank_fusion(merged, k=60)
    deduped = deduplicate_by_content_hash(fused)[:RERANK_FUSED_K]
    timings["fusion_ms"] = (time.perf_counter() - t1) * 1000

    t2 = time.perf_counter()
    reranked = batch_llm_rerank(query_text, deduped)
    timings["reranking_ms"] = (time.perf_counter() - t2) * 1000

    t3 = time.perf_counter()
    final_chunks, stats = finalize_context(query_text, reranked, keep_top_n=keep_top_n)
    sources_block = format_sources(final_chunks)
    system_prompt = f"""You answer questions using ONLY the retrieved sources below.
Cite claims using the matching [n] source number.
If the sources do not contain the answer, say so plainly.

Retrieved sources:
{sources_block}"""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query_text},
        ],
    )
    timings["generation_ms"] = (time.perf_counter() - t3) * 1000

    timings["total_ms"] = sum(timings.values())
    return response.choices[0].message.content, timings


# answer, timings = timed_end_to_end("What did John study and where?")
# print(timings)
# print(answer)

## 10. Add a reranking threshold

> If no candidate receives a sufficiently strong reranker score, tell the
> generation layer that the evidence may be insufficient. Do not force the model
> to answer from weak documents.

In [41]:
RERANK_MIN_SCORE = 5  # on the 0-10 scale used by score_single_pair / batch_llm_rerank

def answer_with_rerank(user_question, keep_top_n=5, min_score=RERANK_MIN_SCORE):
    """Full pipeline: hybrid retrieve -> rerank -> threshold check -> generate.

    TODO:
      1. candidates = get_rerank_candidates(user_question)
      2. reranked = batch_llm_rerank(user_question, candidates)
      3. If reranked is empty OR reranked[0]["reranker_score"] < min_score:
           - do NOT call the generation model with weak evidence
           - return a clear "insufficient evidence" response instead, e.g.
             "I couldn't find strong enough evidence in the documents to answer this."
      4. Otherwise, finalize_context() -> format_sources() -> generate the answer
         the same way Section 9's timed_end_to_end() does, and return it.
    """
    raise NotImplementedError("Implement the threshold check before generation")

# STEP 9 - CHUNKING STRATEGIES 

## Step 9.1 — Install document loading dependencies

We need different libraries to extract text from different document formats:

- Markdown → `markdown`
- HTML → `beautifulsoup4`
- PDF → `pypdf`

The goal is to normalize all formats into a common structure containing:
- extracted text
- source metadata
- file type
- page information where available

In [42]:
from pathlib import Path
import re

from bs4 import BeautifulSoup
from pypdf import PdfReader
import markdown

## Step 9.3 — Normalize extracted text

Different document formats produce text with different whitespace and formatting.

We normalize the extracted text so that downstream chunking receives a consistent representation.

The normalizer will:

1. Convert different newline formats to `\n`
2. Remove excessive spaces
3. Remove excessive blank lines
4. Strip whitespace from the beginning and end

In [43]:
def normalize_text(text):
    """Normalize extracted text into a consistent format."""

    # Normalize newline characters
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # Replace multiple spaces/tabs with a single space
    text = re.sub(r"[ \t]+", " ", text)

    # Remove spaces around newlines
    text = re.sub(r" *\n *", "\n", text)

    # Collapse 3+ newlines into 2
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [44]:
sample = """
John Doe


has worked at ByteMage.

He is a software engineer.
"""

print(normalize_text(sample))

John Doe

has worked at ByteMage.

He is a software engineer.


In [45]:
def load_markdown(file_path):
    """Load a Markdown file and return normalized text + metadata."""

    file_path = Path(file_path)

    markdown_text = file_path.read_text(encoding="utf-8")

    # Convert Markdown → HTML
    html = markdown.markdown(markdown_text)

    # Convert HTML → plain text
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text("\n")

    return {
        "text": normalize_text(text),
        "metadata": {
            "source": file_path.name,
            "file_type": "markdown",
        }
    }

In [47]:
result = load_markdown("../data/sample.md")

print(result["text"])
print(result["metadata"])

ByteMage Employee Knowledge Base

A fictional sample document for experimenting with document parsing, chunking, overlap, metadata, lexical retrieval, vector retrieval, hybrid search, and reranking.

1. Company Overview

ByteMage is a fictional software company that builds cloud applications, data platforms, and AI-powered business tools. The company operates engineering, product, sales, finance, and customer-success departments. Its engineering organization focuses on reliable APIs, scalable web applications, search systems, and machine-learning infrastructure.

2. Employee Profile

John Doe is a Senior Software Engineer in the AI Platform department. He joined ByteMage in March 2022 and works primarily on retrieval-augmented generation systems, search infrastructure, and internal developer tools. John's employee ID is BM-1047 and his manager is Sarah Ahmed.

3. Education and Background

John completed a Bachelor of Science in Computer Science from the University of Lahore in 2018. Be

In [48]:
def load_html(file_path):
    """Load an HTML file, remove non-content elements, and return normalized text."""

    file_path = Path(file_path)

    html = file_path.read_text(encoding="utf-8")

    soup = BeautifulSoup(html, "html.parser")

    # Remove elements that are not useful for retrieval
    for tag in soup([
        "script",
        "style",
        "nav",
        "header",
        "footer",
        "aside"
    ]):
        tag.decompose()

    # Extract visible text
    text = soup.get_text("\n")

    return {
        "text": normalize_text(text),
        "metadata": {
            "source": file_path.name,
            "file_type": "html",
        }
    }

In [49]:
result = load_html("../data/sample.html")

print(result["text"])
print(result["metadata"])

ByteMage Employee Knowledge Base

ByteMage Employee Knowledge Base

A fictional sample document for experimenting with document parsing, chunking, overlap, metadata, lexical retrieval, vector retrieval, hybrid search, and reranking.

1. Company Overview

ByteMage is a fictional software company that builds cloud applications, data platforms, and AI-powered business tools. The company operates engineering, product, sales, finance, and customer-success departments. Its engineering organization focuses on reliable APIs, scalable web applications, search systems, and machine-learning infrastructure.

2. Employee Profile

John Doe is a Senior Software Engineer in the AI Platform department. He joined ByteMage in March 2022 and works primarily on retrieval-augmented generation systems, search infrastructure, and internal developer tools. John's employee ID is BM-1047 and his manager is Sarah Ahmed.

3. Education and Background

John completed a Bachelor of Science in Computer Science from th

In [50]:
def load_pdf(file_path):
    """Load a PDF page-by-page while preserving page numbers."""

    file_path = Path(file_path)

    reader = PdfReader(str(file_path))

    results = []

    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text() or ""

        text = normalize_text(text)

        if not text:
            continue

        results.append({
            "text": text,
            "metadata": {
                "source": file_path.name,
                "file_type": "pdf",
                "page": page_number,
            }
        })

    return results

In [51]:
results = load_pdf("../data/sample.pdf")

for result in results:
    print("PAGE:", result["metadata"]["page"])
    print(result["text"][:300])
    print("-" * 50)

PAGE: 1
ByteMage Employee Knowledge Base
A fictional sample document for experimenting with document parsing, chunking, overlap, metadata, lexical
retrieval, vector retrieval, hybrid search, and reranking.
1. Company Overview
ByteMage is a fictional software company that builds cloud applications, data plat
--------------------------------------------------
PAGE: 2
7. Retrieval Example
Consider the question: Where did John study? A semantic retriever may return passages discussing John's
education because they are conceptually related. A lexical retriever may prioritize the exact terms John, study,
Computer Science, or University. The hybrid system merges cand
--------------------------------------------------


## Step 9.8 — Unified document loader

The retrieval pipeline should not need to know how each file format is parsed.

The unified loader detects the file extension and dispatches the file to the appropriate format-specific loader.

In [52]:
def load_document(file_path):
    """Load a document using the appropriate format-specific loader."""

    file_path = Path(file_path)

    extension = file_path.suffix.lower()

    if extension in [".md", ".markdown"]:
        result = load_markdown(file_path)
        return [result]

    elif extension in [".html", ".htm"]:
        result = load_html(file_path)
        return [result]

    elif extension == ".pdf":
        return load_pdf(file_path)

    else:
        raise ValueError(
            f"Unsupported file type: {extension}"
        )

In [54]:
load_document("../data/sample.pdf")

[{'text': "ByteMage Employee Knowledge Base\nA fictional sample document for experimenting with document parsing, chunking, overlap, metadata, lexical\nretrieval, vector retrieval, hybrid search, and reranking.\n1. Company Overview\nByteMage is a fictional software company that builds cloud applications, data platforms, and AI-powered\nbusiness tools. The company operates engineering, product, sales, finance, and customer-success\ndepartments. Its engineering organization focuses on reliable APIs, scalable web applications, search systems,\nand machine-learning infrastructure.\n2. Employee Profile\nJohn Doe is a Senior Software Engineer in the AI Platform department. He joined ByteMage in March 2022 and\nworks primarily on retrieval-augmented generation systems, search infrastructure, and internal developer tools.\nJohn's employee ID is BM-1047 and his manager is Sarah Ahmed.\n3. Education and Background\nJohn completed a Bachelor of Science in Computer Science from the University of L

In [56]:
files = [
    "../data/sample.md",
    "../data/sample.html",
    "../data/sample.pdf",
]

for file in files:

    print("=" * 70)
    print("FILE:", file)
    print("=" * 70)

    documents = load_document(file)

    for document in documents:

        print(document["metadata"])
        print(document["text"][:500])
        print()

FILE: ../data/sample.md
{'source': 'sample.md', 'file_type': 'markdown'}
ByteMage Employee Knowledge Base

A fictional sample document for experimenting with document parsing, chunking, overlap, metadata, lexical retrieval, vector retrieval, hybrid search, and reranking.

1. Company Overview

ByteMage is a fictional software company that builds cloud applications, data platforms, and AI-powered business tools. The company operates engineering, product, sales, finance, and customer-success departments. Its engineering organization focuses on reliable APIs, scalable we

FILE: ../data/sample.html
{'source': 'sample.html', 'file_type': 'html'}
ByteMage Employee Knowledge Base

ByteMage Employee Knowledge Base

A fictional sample document for experimenting with document parsing, chunking, overlap, metadata, lexical retrieval, vector retrieval, hybrid search, and reranking.

1. Company Overview

ByteMage is a fictional software company that builds cloud applications, data platforms, and AI-p

In [57]:
def load_directory(directory):
    """Load all supported documents from a directory."""

    directory = Path(directory)

    supported_extensions = {
        ".md",
        ".markdown",
        ".html",
        ".htm",
        ".pdf",
    }

    all_documents = []

    for file_path in directory.iterdir():

        if not file_path.is_file():
            continue

        if file_path.suffix.lower() not in supported_extensions:
            continue

        documents = load_document(file_path)

        all_documents.extend(documents)

    return all_documents

In [59]:
documents = load_directory("../data")

print("Number of loaded documents:", len(documents))

Number of loaded documents: 4


In [60]:
for i, document in enumerate(documents):

    print("=" * 70)
    print("DOCUMENT:", i)
    print("METADATA:", document["metadata"])
    print("TEXT:", document["text"][:300])

DOCUMENT: 0
METADATA: {'source': 'sample.md', 'file_type': 'markdown'}
TEXT: ByteMage Employee Knowledge Base

A fictional sample document for experimenting with document parsing, chunking, overlap, metadata, lexical retrieval, vector retrieval, hybrid search, and reranking.

1. Company Overview

ByteMage is a fictional software company that builds cloud applications, data p
DOCUMENT: 1
METADATA: {'source': 'sample.html', 'file_type': 'html'}
TEXT: ByteMage Employee Knowledge Base

ByteMage Employee Knowledge Base

A fictional sample document for experimenting with document parsing, chunking, overlap, metadata, lexical retrieval, vector retrieval, hybrid search, and reranking.

1. Company Overview

ByteMage is a fictional software company that
DOCUMENT: 2
METADATA: {'source': 'sample.pdf', 'file_type': 'pdf', 'page': 1}
TEXT: ByteMage Employee Knowledge Base
A fictional sample document for experimenting with document parsing, chunking, overlap, metadata, lexical
retrieval, vector retri

In [2]:
# Load document & split into paragraphs
document_path = PROJECT_ROOT / "data" / "profile.txt"
document_text = document_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in document_text.split("\n\n") if p.strip()]
documents = [{"id": i, "text": p} for i, p in enumerate(paragraphs)]
print(f"Loaded {len(documents)} chunks.")

Loaded 22 chunks.


## 3. Fixed-Character Chunking

Fixed-character chunking splits a document into chunks based on a fixed
number of characters.

Example:
- Chunk size: 1000 characters
- Overlap: 200 characters

The overlap helps preserve context between neighboring chunks.

We also store the character start and end offsets so that each chunk can
be traced back to its original location in the document.

In [7]:
def fixed_character_chunking(text, chunk_size=1000, overlap=200):
    """
    Split text into fixed-size character chunks with optional overlap.

    Returns:
        List of dictionaries containing:
        - chunk_id
        - text
        - start_char
        - end_char
    """

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than 0")

    if overlap < 0:
        raise ValueError("overlap cannot be negative")

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []

    start = 0
    chunk_number = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        chunk_text = text[start:end]

        chunks.append({
            "chunk_id": f"char-{chunk_number:03d}",
            "text": chunk_text,
            "start_char": start,
            "end_char": end,
        })

        chunk_number += 1
          # IMPORTANT:
        # Stop when we reach the end of the document.
        if end == len(text):
            break

        # Move forward while keeping overlap
        start = end - overlap

    return chunks

In [8]:
text = document_text

char_chunks = fixed_character_chunking(
    text,
    chunk_size=1000,
    overlap=200
)

print("Number of chunks:", len(char_chunks))

print("\nFirst chunk:")
print(char_chunks[0])

Number of chunks: 11

First chunk:
{'chunk_id': 'char-000', 'text': "Fictional Biography of John Doe\nJohn Doe: A Life of Curiosity, Innovation, and Service\n\nJohn Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, John displayed an unusual curiosity about how things worked. Whether dismantling old radios in his parents' garage or spending hours reading books from the local library, he developed a passion for learning that would shape the rest of his life.\n\nJohn was the eldest of three children born to Michael and Sarah Doe. His father worked as a mechanical engineer, while his mother was a high school English teacher. Growing up in a household that valued both analytical thinking and creativity, John learned the importance of balancing logic with imagination. Family evenings often consisted of discussions about science, literatur

## 4. Token-Based Chunking

Token-based chunking splits text according to the number of tokens
rather than the number of characters.

Example:
- Chunk size: 300 tokens
- Overlap: 50 tokens

This gives more predictable control over the amount of text sent to
embedding models or LLMs.

The chunks are reconstructed by decoding the token slices back into text.

In [3]:
import tiktoken
encoding = tiktoken.get_encoding("cl100k_base")

In [4]:
def token_based_chunking(text, chunk_size=300, overlap=50, encoding=None):
    """
    Split text into chunks based on token count.

    Args:
        text: Input text.
        chunk_size: Maximum number of tokens per chunk.
        overlap: Number of overlapping tokens.
        encoding: tiktoken encoding.

    Returns:
        List of dictionaries containing:
        - chunk_id
        - text
        - start_token
        - end_token
        - token_count
    """

    if encoding is None:
        encoding = tiktoken.get_encoding("cl100k_base")

    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than 0")

    if overlap < 0:
        raise ValueError("overlap cannot be negative")

    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    tokens = encoding.encode(text)

    chunks = []

    start = 0
    chunk_number = 0

    while start < len(tokens):

        end = min(start + chunk_size, len(tokens))

        chunk_tokens = tokens[start:end]

        chunk_text = encoding.decode(chunk_tokens)

        chunks.append({
            "chunk_id": f"token-{chunk_number:03d}",
            "text": chunk_text,
            "start_token": start,
            "end_token": end,
            "token_count": len(chunk_tokens),
        })

        chunk_number += 1

        # IMPORTANT:
        # Stop when we reach the end of the document.
        if end == len(tokens):
            break

        start = end - overlap

    return chunks

In [5]:
token_chunks = token_based_chunking(
    document_text,
    chunk_size=300,
    overlap=50,
    encoding=encoding
)

print("Number of chunks:", len(token_chunks))

print("\nFirst chunk:")
print(token_chunks[0])

Number of chunks: 6

First chunk:
{'chunk_id': 'token-000', 'text': "Fictional Biography of John Doe\nJohn Doe: A Life of Curiosity, Innovation, and Service\n\nJohn Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, John displayed an unusual curiosity about how things worked. Whether dismantling old radios in his parents' garage or spending hours reading books from the local library, he developed a passion for learning that would shape the rest of his life.\n\nJohn was the eldest of three children born to Michael and Sarah Doe. His father worked as a mechanical engineer, while his mother was a high school English teacher. Growing up in a household that valued both analytical thinking and creativity, John learned the importance of balancing logic with imagination. Family evenings often consisted of discussions about science, literatur

In [6]:
for chunk in token_chunks[:5]:
    print(
        chunk["chunk_id"],
        "tokens:", chunk["token_count"]
    )

token-000 tokens: 300
token-001 tokens: 300
token-002 tokens: 300
token-003 tokens: 300
token-004 tokens: 300


## 5. Paragraph-Based Chunking

Paragraph-based chunking groups complete paragraphs together until a
maximum token limit is reached.

A paragraph is not split unless it is larger than the maximum chunk size.

This preserves the natural semantic boundaries of the document better
than blindly splitting at a fixed character or token position.

In [7]:
def paragraph_based_chunking(
    paragraphs,
    max_tokens=300,
    encoding=None
):
    """
    Group complete paragraphs until max_tokens is reached.

    A paragraph is kept intact unless it is larger than max_tokens.
    Oversized paragraphs are split using token-based chunking.
    """

    if encoding is None:
        encoding = tiktoken.get_encoding("cl100k_base")

    chunks = []
    current_paragraphs = []
    current_token_count = 0
    chunk_number = 0

    def save_current_chunk():
        nonlocal current_paragraphs
        nonlocal current_token_count
        nonlocal chunk_number

        if not current_paragraphs:
            return

        chunk_text = "\n\n".join(current_paragraphs)

        chunks.append({
            "chunk_id": f"paragraph-{chunk_number:03d}",
            "text": chunk_text,
            "token_count": current_token_count,
        })

        chunk_number += 1
        current_paragraphs = []
        current_token_count = 0

    for paragraph in paragraphs:

        paragraph = paragraph.strip()

        if not paragraph:
            continue

        paragraph_tokens = encoding.encode(paragraph)
        paragraph_token_count = len(paragraph_tokens)

        # If paragraph itself is too large,
        # save current chunk first and split paragraph.
        if paragraph_token_count > max_tokens:

            save_current_chunk()

            oversized_chunks = token_based_chunking(
                paragraph,
                chunk_size=max_tokens,
                overlap=0,
                encoding=encoding
            )

            for oversized_chunk in oversized_chunks:
                chunks.append({
                    "chunk_id": f"paragraph-{chunk_number:03d}",
                    "text": oversized_chunk["text"],
                    "token_count": oversized_chunk["token_count"],
                })

                chunk_number += 1

            continue

        # If adding this paragraph would exceed the limit,
        # save the current chunk first.
        if (
            current_token_count + paragraph_token_count > max_tokens
            and current_paragraphs
        ):
            save_current_chunk()

        current_paragraphs.append(paragraph)
        current_token_count += paragraph_token_count

    # Save remaining paragraphs
    save_current_chunk()

    return chunks

In [8]:
paragraphs = [
    p.strip()
    for p in document_text.split("\n\n")
    if p.strip()
]

In [9]:
paragraph_chunks = paragraph_based_chunking(
    paragraphs,
    max_tokens=300,
    encoding=encoding
)

In [10]:
print("Number of chunks:", len(paragraph_chunks))

for chunk in paragraph_chunks[:3]:
    print("=" * 80)
    print(chunk["chunk_id"])
    print("Tokens:", chunk["token_count"])
    print(chunk["text"])

Number of chunks: 6
paragraph-000
Tokens: 251
Fictional Biography of John Doe
John Doe: A Life of Curiosity, Innovation, and Service

John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, John displayed an unusual curiosity about how things worked. Whether dismantling old radios in his parents' garage or spending hours reading books from the local library, he developed a passion for learning that would shape the rest of his life.

John was the eldest of three children born to Michael and Sarah Doe. His father worked as a mechanical engineer, while his mother was a high school English teacher. Growing up in a household that valued both analytical thinking and creativity, John learned the importance of balancing logic with imagination. Family evenings often consisted of discussions about science, literature, current events, and futur

## 6. Heading-Aware Chunking

Heading-aware chunking preserves the document's structural hierarchy.

For Markdown documents, headings are detected from `#`, `##`, `###`, etc.

Each chunk stores the heading path that describes where the content
appears in the document.

Example:

Leave Policy
    → Sick Leave
        → Documentation

The heading path can also optionally be prepended to the chunk text
before embedding.

In [11]:
import re
def get_markdown_heading(line):
    """
    Return (heading_level, heading_text) if the line is a Markdown heading.
    Otherwise return None.
    """

    match = re.match(r"^(#{1,6})\s+(.+?)\s*$", line)

    if not match:
        return None

    level = len(match.group(1))
    heading = match.group(2).strip()

    return level, heading

In [12]:
test_lines = [
    "# Leave Policy",
    "## Sick Leave",
    "### Documentation",
    "This is normal text"
]

for line in test_lines:
    print(line, "->", get_markdown_heading(line))

# Leave Policy -> (1, 'Leave Policy')
## Sick Leave -> (2, 'Sick Leave')
### Documentation -> (3, 'Documentation')
This is normal text -> None


In [13]:
def heading_aware_chunking(
    text,
    max_tokens=300,
    encoding=None,
    prepend_heading=True
):
    """
    Split Markdown text into chunks while preserving heading hierarchy.
    """

    if encoding is None:
        encoding = tiktoken.get_encoding("cl100k_base")

    lines = text.splitlines()

    chunks = []

    heading_stack = []
    current_content = []

    def flush_content():
        nonlocal current_content

        if not current_content:
            return

        content = "\n".join(current_content).strip()

        if not content:
            current_content = []
            return

        heading_path = [item["text"] for item in heading_stack]

        if prepend_heading and heading_path:
            heading_text = " > ".join(heading_path)
            content_for_embedding = (
                f"Section: {heading_text}\n\n{content}"
            )
        else:
            content_for_embedding = content

        token_count = len(encoding.encode(content_for_embedding))

        # If content fits
        if token_count <= max_tokens:

            chunks.append({
                "chunk_id": f"heading-{len(chunks):03d}",
                "text": content_for_embedding,
                "heading_path": heading_path.copy(),
                "token_count": token_count,
            })

        else:
            # If section is too large, fall back to token chunking
            split_chunks = token_based_chunking(
                content_for_embedding,
                chunk_size=max_tokens,
                overlap=50,
                encoding=encoding
            )

            for split_chunk in split_chunks:
                chunks.append({
                    "chunk_id": f"heading-{len(chunks):03d}",
                    "text": split_chunk["text"],
                    "heading_path": heading_path.copy(),
                    "token_count": split_chunk["token_count"],
                })

        current_content = []

    for line in lines:

        heading = get_markdown_heading(line)

        if heading:

            # Save content belonging to previous heading
            flush_content()

            level, heading_text = heading

            # Remove headings at the same or deeper level
            heading_stack = [
                item
                for item in heading_stack
                if item["level"] < level
            ]

            heading_stack.append({
                "level": level,
                "text": heading_text
            })

        else:
            current_content.append(line)

    # Save remaining content
    flush_content()

    return chunks

In [14]:
heading_chunks = heading_aware_chunking(
    document_text,
    max_tokens=300,
    encoding=encoding,
    prepend_heading=True
)

In [15]:
print("Number of chunks:", len(heading_chunks))

for chunk in heading_chunks[:5]:
    print("=" * 80)
    print("ID:", chunk["chunk_id"])
    print("Heading:", chunk["heading_path"])
    print("Tokens:", chunk["token_count"])
    print(chunk["text"][:500])

Number of chunks: 6
ID: heading-000
Heading: []
Tokens: 300
Fictional Biography of John Doe
John Doe: A Life of Curiosity, Innovation, and Service

John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, John displayed an unusual curiosity about how things worked. Whether dismantling old radios in his parents' garage or spending hours reading books from the local library, he developed a passion for lea
ID: heading-001
Heading: []
Tokens: 300
 assignments.

By the time he entered high school, John had developed a strong interest in computer programming. He spent countless evenings teaching himself programming languages through books and online tutorials. His first meaningful project was a simple budgeting application he created to help his family track monthly expenses. Although basic by today's standards, the project introduced him t

In [16]:
print("Document characters:", len(document_text))
print("Document tokens:", len(encoding.encode(document_text)))
print()
print("Fixed-character chunks:", len(char_chunks))
print("Token chunks:", len(token_chunks))
print("Paragraph chunks:", len(paragraph_chunks))
print("Heading-aware chunks:", len(heading_chunks))

Document characters: 8617
Document tokens: 1392



NameError: name 'char_chunks' is not defined

In [17]:
print("=" * 80)
print("FIXED CHARACTER")
print(char_chunks[0]["text"][:500])

print("=" * 80)
print("TOKEN BASED")
print(token_chunks[0]["text"][:500])

print("=" * 80)
print("PARAGRAPH BASED")
print(paragraph_chunks[0]["text"][:500])

print("=" * 80)
print("HEADING AWARE")
print(heading_chunks[0]["text"][:500])

FIXED CHARACTER


NameError: name 'char_chunks' is not defined

## 7. Recursive Splitting

Recursive splitting tries a prioritized list of separators, from broadest to
narrowest: section, paragraph, sentence, word, character.

It only moves to a smaller separator when the current unit still exceeds the
token limit after splitting on the current separator. This avoids
fixed-character chunking's mid-word cuts while still guaranteeing every chunk
fits the token budget, even for pathological input (e.g. one giant
paragraph with no punctuation).

In [18]:
SEPARATORS = [
    ("section", "\n\n\n"),
    ("paragraph", "\n\n"),
    ("sentence", ". "),
    ("word", " "),
    ("character", ""),
]


def recursive_split(text, max_tokens=300, encoding=None, separators=None):
    """
    Recursively split text using a prioritized list of separators.

    Returns:
        List of dictionaries containing:
        - chunk_id
        - text
        - token_count
    """

    if encoding is None:
        encoding = tiktoken.get_encoding("cl100k_base")

    if separators is None:
        separators = [sep for _, sep in SEPARATORS]

    def token_count(candidate_text):
        return len(encoding.encode(candidate_text))

    def split_recursive(chunk_text, remaining_separators):

        if token_count(chunk_text) <= max_tokens:
            return [chunk_text]

        if not remaining_separators:
            # No separators left — fall back to a hard token-level split.
            hard_chunks = token_based_chunking(
                chunk_text,
                chunk_size=max_tokens,
                overlap=0,
                encoding=encoding,
            )
            return [c["text"] for c in hard_chunks]

        separator = remaining_separators[0]
        next_separators = remaining_separators[1:]

        parts = list(chunk_text) if separator == "" else chunk_text.split(separator)

        pieces = []
        current = ""

        for part in parts:
            candidate = current + (separator if current else "") + part

            if token_count(candidate) <= max_tokens:
                current = candidate
                continue

            if current:
                pieces.append(current)
                current = ""

            if token_count(part) > max_tokens:
                pieces.extend(split_recursive(part, next_separators))
            else:
                current = part

        if current:
            pieces.append(current)

        return pieces

    raw_pieces = split_recursive(text, separators)

    chunks = []
    for piece in raw_pieces:
        piece = piece.strip()

        if not piece:
            continue

        chunks.append({
            "chunk_id": f"recursive-{len(chunks):03d}",
            "text": piece,
            "token_count": token_count(piece),
        })

    return chunks

In [19]:
recursive_chunks = recursive_split(
    document_text,
    max_tokens=300,
    encoding=encoding,
)

print("Number of chunks:", len(recursive_chunks))

for chunk in recursive_chunks[:3]:
    print("=" * 80)
    print(chunk["chunk_id"], "- tokens:", chunk["token_count"])
    print(chunk["text"][:300])

Number of chunks: 6
recursive-000 - tokens: 252
Fictional Biography of John Doe
John Doe: A Life of Curiosity, Innovation, and Service

John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, J
recursive-001 - tokens: 253
By the time he entered high school, John had developed a strong interest in computer programming. He spent countless evenings teaching himself programming languages through books and online tutorials. His first meaningful project was a simple budgeting application he created to help his family track
recursive-002 - tokens: 269
Upon completing his degree in 2009, John accepted a position as a junior software engineer at Innovate Solutions, a midsized technology company specializing in enterprise software. In this role, he worked on internal productivity tools, customer relationship management systems, and data visual

## 8. Experiment with Overlap

We compare three overlap settings on token-based chunking: none, small, and
large. A small `chunk_size` (80 tokens) is used deliberately so that
sentence-level facts are likely to straddle a chunk boundary, making the
effect of overlap visible.

For each configuration we measure:
- Number of chunks
- Total embedding tokens (chunks are sent to the embedding model, so overlap
  directly increases cost)
- Duplicated tokens — `total_embedding_tokens - unique_document_tokens`
- Context redundancy — the duplication ratio

We also check whether a fact that sits near a chunk boundary survives inside
a single chunk. This is a cheap proxy for retrieval recall: if the whole fact
never appears in one chunk, no retriever can return it as one coherent
piece of evidence.

In [20]:
def answer_found_in_single_chunk(chunks, answer_substring):
    """Return True if a single chunk fully contains answer_substring (case-insensitive)."""
    target = answer_substring.lower()
    return any(target in chunk["text"].lower() for chunk in chunks)


OVERLAP_TEST_CHUNK_SIZE = 80  # small on purpose, to make boundary effects visible

overlap_configs = [
    ("no_overlap", 0),
    ("small_overlap", 20),
    ("large_overlap", 60),
]

# Construct — rather than guess — a fact that straddles a no-overlap chunk
# boundary. Token decoding concatenates per-token byte sequences, so slicing
# 6 tokens on either side of a chunk-size multiple is guaranteed to land
# across two adjacent no-overlap chunks.
document_tokens = encoding.encode(document_text)
boundary_index = OVERLAP_TEST_CHUNK_SIZE * 3
boundary_sensitive_fact = encoding.decode(
    document_tokens[boundary_index - 6: boundary_index + 6]
).strip()

print("Boundary-sensitive fact (straddles a no-overlap chunk edge):")
print(repr(boundary_sensitive_fact))
print()

unique_document_tokens = len(document_tokens)

overlap_results = []

for label, overlap in overlap_configs:

    chunks = token_based_chunking(
        document_text,
        chunk_size=OVERLAP_TEST_CHUNK_SIZE,
        overlap=overlap,
        encoding=encoding,
    )

    total_tokens = sum(chunk["token_count"] for chunk in chunks)
    duplicated_tokens = total_tokens - unique_document_tokens
    duplication_ratio = round(duplicated_tokens / total_tokens, 3) if total_tokens else 0.0

    overlap_results.append({
        "config": label,
        "overlap": overlap,
        "num_chunks": len(chunks),
        "total_embedding_tokens": total_tokens,
        "duplicated_tokens": duplicated_tokens,
        "duplication_ratio": duplication_ratio,
        "fact_preserved_in_one_chunk": answer_found_in_single_chunk(chunks, boundary_sensitive_fact),
    })

for row in overlap_results:
    print(row)

Boundary-sensitive fact (straddles a no-overlap chunk edge):
'often described him as thoughtful, disciplined, and eager to help'

{'config': 'no_overlap', 'overlap': 0, 'num_chunks': 18, 'total_embedding_tokens': 1392, 'duplicated_tokens': 0, 'duplication_ratio': 0.0, 'fact_preserved_in_one_chunk': False}
{'config': 'small_overlap', 'overlap': 20, 'num_chunks': 23, 'total_embedding_tokens': 1832, 'duplicated_tokens': 440, 'duplication_ratio': 0.24, 'fact_preserved_in_one_chunk': True}
{'config': 'large_overlap', 'overlap': 60, 'num_chunks': 67, 'total_embedding_tokens': 5352, 'duplicated_tokens': 3960, 'duplication_ratio': 0.74, 'fact_preserved_in_one_chunk': True}


## 9. Create Boundary-Crossing Questions

We write questions in four categories:

- **fits_in_one_chunk** — the answer is short and local, should survive any
  reasonable chunking strategy.
- **begins_end_continues_next** — the answer sits near a chunk boundary in
  small fixed-size chunks and may be split across two chunks.
- **requires_heading_and_body** — the answer only makes sense together with
  its section heading (tested against `sample.md`, which has real headings —
  `profile.txt` does not).
- **requires_two_sections** — the full answer is only assembled by reading
  two distant sections; no single chunk should ever contain it.

For the heading-based questions we run `heading_aware_chunking` against the
raw Markdown source (`sample.md`) instead of `profile.txt`, since
`profile.txt` has no headings to preserve.

In [21]:
sample_md_text = Path("../data/sample.md").read_text(encoding="utf-8")

sample_md_heading_chunks = heading_aware_chunking(
    sample_md_text,
    max_tokens=120,
    encoding=encoding,
    prepend_heading=True,
)

print("Number of sample.md heading-aware chunks:", len(sample_md_heading_chunks))
for chunk in sample_md_heading_chunks[:3]:
    print("-", chunk["chunk_id"], chunk["heading_path"])

Number of sample.md heading-aware chunks: 11
- heading-000 ['ByteMage Employee Knowledge Base']
- heading-001 ['ByteMage Employee Knowledge Base', '1. Company Overview']
- heading-002 ['ByteMage Employee Knowledge Base', '2. Employee Profile']


In [22]:
def evaluate_boundary_question(chunks, required_substrings):
    """
    Check how many chunks the required substrings are spread across.

    Returns:
        {
            "all_substrings_found": bool,
            "fully_contained_in_one_chunk": bool,
            "matches": [(substring, [chunk_id, ...]), ...]
        }
    """

    matches = []

    for substring in required_substrings:
        substring_lower = substring.lower()
        found_in = [
            chunk["chunk_id"]
            for chunk in chunks
            if substring_lower in chunk["text"].lower()
        ]
        matches.append((substring, found_in))

    all_found = all(found for _, found in matches)

    fully_contained_in_one_chunk = False
    if all_found:
        common_chunks = set(matches[0][1])
        for _, found in matches[1:]:
            common_chunks &= set(found)
        fully_contained_in_one_chunk = len(common_chunks) > 0

    return {
        "all_substrings_found": all_found,
        "fully_contained_in_one_chunk": fully_contained_in_one_chunk,
        "matches": matches,
    }


boundary_questions = [
    {
        "category": "fits_in_one_chunk",
        "question": "What is John Doe's employee ID?",
        "required_substrings": ["BM-1047"],
        "chunk_source": "sample_md_heading_chunks",
    },
    {
        "category": "begins_end_continues_next",
        "question": "Where and why did John co-found BrightPath Technologies?",
        "required_substrings": ["co-founded a fictional startup called BrightPath Technologies"],
        "chunk_source": "char_chunks",
    },
    {
        "category": "requires_heading_and_body",
        "question": "What is ByteMage's default chunk size and overlap?",
        "required_substrings": ["400 tokens per chunk with an overlap of 60 tokens"],
        "chunk_source": "sample_md_heading_chunks",
    },
    {
        "category": "requires_two_sections",
        "question": "What is John Doe's employee ID, and what chunk overlap does ByteMage use by default?",
        "required_substrings": ["BM-1047", "overlap of 60 tokens"],
        "chunk_source": "sample_md_heading_chunks",
    },
]

chunk_sources = {
    "char_chunks": char_chunks,
    "sample_md_heading_chunks": sample_md_heading_chunks,
}

for item in boundary_questions:
    chunks = chunk_sources[item["chunk_source"]]
    result = evaluate_boundary_question(chunks, item["required_substrings"])

    print("=" * 80)
    print("Category:", item["category"])
    print("Question:", item["question"])
    print("Chunk source:", item["chunk_source"])
    print("All parts found:", result["all_substrings_found"])
    print("Fully contained in one chunk:", result["fully_contained_in_one_chunk"])
    for substring, found_in in result["matches"]:
        print(f"  '{substring}' -> {found_in}")

NameError: name 'char_chunks' is not defined

## 10. Implement Parent-Child Retrieval

Small chunks retrieve well (a focused embedding matches a focused query) but
can be too narrow to generate a good answer from. Parent-child retrieval
splits the problem in two:

- **Child chunks** — small, used for embedding and retrieval.
- **Parent chunks** — larger paragraph-grouped sections, used for generation.

Each child stores its `parent_id`. When a child is retrieved, we look up its
parent and hand the larger parent text to the generation model instead of the
narrow child text — deduplicating parents when several retrieved children map
to the same section.

In [23]:
def create_parent_child_chunks(text, parent_max_tokens=400, child_max_tokens=120, encoding=None):
    """
    Build a two-level chunk hierarchy.

    Returns:
        parent_chunks: list of larger paragraph-grouped chunks (for generation)
        child_chunks: list of smaller token-based chunks, each with a parent_id
                      (for embedding/retrieval)
        parent_lookup: dict mapping parent_id -> parent chunk
    """

    if encoding is None:
        encoding = tiktoken.get_encoding("cl100k_base")

    text_paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    parent_chunks = paragraph_based_chunking(
        text_paragraphs,
        max_tokens=parent_max_tokens,
        encoding=encoding,
    )

    for i, parent in enumerate(parent_chunks):
        parent["parent_id"] = f"parent-{i:03d}"

    child_chunks = []

    for parent in parent_chunks:
        children = token_based_chunking(
            parent["text"],
            chunk_size=child_max_tokens,
            overlap=20,
            encoding=encoding,
        )

        for child in children:
            child_chunks.append({
                "chunk_id": f"{parent['parent_id']}-child-{len(child_chunks):03d}",
                "text": child["text"],
                "token_count": child["token_count"],
                "parent_id": parent["parent_id"],
            })

    parent_lookup = {parent["parent_id"]: parent for parent in parent_chunks}

    return parent_chunks, child_chunks, parent_lookup


def resolve_children_to_parents(retrieved_child_chunks, parent_lookup):
    """Given retrieved child chunks, return deduplicated parent chunks for generation."""

    seen_parent_ids = set()
    resolved_parents = []

    for child in retrieved_child_chunks:
        parent_id = child["parent_id"]

        if parent_id in seen_parent_ids:
            continue

        seen_parent_ids.add(parent_id)
        resolved_parents.append(parent_lookup[parent_id])

    return resolved_parents

In [24]:
parent_chunks, child_chunks, parent_lookup = create_parent_child_chunks(
    document_text,
    parent_max_tokens=400,
    child_max_tokens=120,
    encoding=encoding,
)

print("Parent chunks:", len(parent_chunks))
print("Child chunks:", len(child_chunks))

# Simulate retrieval returning a handful of child chunks, possibly from the
# same parent section.
example_retrieved_children = child_chunks[3:6]

print("\nRetrieved child chunks:")
for child in example_retrieved_children:
    print("-", child["chunk_id"], "(parent:", child["parent_id"] + ")")

resolved_parents = resolve_children_to_parents(example_retrieved_children, parent_lookup)

print("\nResolved parents (deduplicated):", len(resolved_parents))
for parent in resolved_parents:
    print("=" * 80)
    print(parent["parent_id"])
    print(parent["text"][:300])

Parent chunks: 4
Child chunks: 15

Retrieved child chunks:
- parent-000-child-003 (parent: parent-000)
- parent-001-child-004 (parent: parent-001)
- parent-001-child-005 (parent: parent-001)

Resolved parents (deduplicated): 2
parent-000
Fictional Biography of John Doe
John Doe: A Life of Curiosity, Innovation, and Service

John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, J
parent-001
During his second year at university, John joined a research lab focused on artificial intelligence and natural language processing. Working alongside professors and graduate students, he contributed to several experimental projects involving intelligent tutoring systems and document analysis. These


## 11. Handle Tables and Structured Content

Flattening a table into plain text without labels destroys the meaning of
each cell (a bare `"BM-1047"` in running text loses the fact that it is an
employee ID). We compare two strategies on a small illustrative employee
table:

- **Row-based chunking** — one chunk per row, values labeled with their
  column names. Retrieves precisely for row-specific queries (e.g. "who
  manages John Doe?") but loses table-wide context.
- **Full-table chunking** — the entire table serialized as one chunk.
  Preserves table-wide context (e.g. "how many employees are in AI
  Platform?") but risks exceeding chunk size limits and dilutes
  row-specific retrieval precision.

In [25]:
sample_table = {
    "columns": ["Employee ID", "Name", "Department", "Manager", "Start Date"],
    "rows": [
        ["BM-1047", "John Doe", "AI Platform", "Sarah Ahmed", "2022-03-01"],
        ["BM-1048", "Maria Chen", "AI Platform", "Sarah Ahmed", "2022-06-15"],
        ["BM-1102", "Ali Raza", "Search Infrastructure", "Sarah Ahmed", "2023-01-10"],
        ["BM-1150", "Priya Nair", "Customer Success", "Tom Becker", "2023-05-20"],
    ],
    "location": "sample.md > Section 6: Search Metadata (illustrative table, not present in the source file)",
}


def table_row_to_text(columns, row):
    """Convert one table row into a labeled text representation."""
    return "; ".join(f"{column}: {value}" for column, value in zip(columns, row))


def chunk_table_by_row(table):
    """Row-based chunking: one chunk per row, with column labels preserved."""

    return [
        {
            "chunk_id": f"table-row-{i:03d}",
            "text": table_row_to_text(table["columns"], row),
            "table_location": table["location"],
            "row_index": i,
        }
        for i, row in enumerate(table["rows"])
    ]


def chunk_table_full(table):
    """Full-table chunking: the entire table serialized as one Markdown-style chunk."""

    header = " | ".join(table["columns"])
    separator = " | ".join(["---"] * len(table["columns"]))
    body_lines = [" | ".join(str(value) for value in row) for row in table["rows"]]

    table_text = "\n".join([header, separator, *body_lines])

    return [{
        "chunk_id": "table-full-000",
        "text": table_text,
        "table_location": table["location"],
    }]

In [26]:
table_row_chunks = chunk_table_by_row(sample_table)
table_full_chunks = chunk_table_full(sample_table)

print("ROW-BASED CHUNKS:")
for chunk in table_row_chunks:
    print(chunk["chunk_id"], "-", chunk["text"])

print("\nFULL-TABLE CHUNK:")
print(table_full_chunks[0]["text"])

ROW-BASED CHUNKS:
table-row-000 - Employee ID: BM-1047; Name: John Doe; Department: AI Platform; Manager: Sarah Ahmed; Start Date: 2022-03-01
table-row-001 - Employee ID: BM-1048; Name: Maria Chen; Department: AI Platform; Manager: Sarah Ahmed; Start Date: 2022-06-15
table-row-002 - Employee ID: BM-1102; Name: Ali Raza; Department: Search Infrastructure; Manager: Sarah Ahmed; Start Date: 2023-01-10
table-row-003 - Employee ID: BM-1150; Name: Priya Nair; Department: Customer Success; Manager: Tom Becker; Start Date: 2023-05-20

FULL-TABLE CHUNK:
Employee ID | Name | Department | Manager | Start Date
--- | --- | --- | --- | ---
BM-1047 | John Doe | AI Platform | Sarah Ahmed | 2022-03-01
BM-1048 | Maria Chen | AI Platform | Sarah Ahmed | 2022-06-15
BM-1102 | Ali Raza | Search Infrastructure | Sarah Ahmed | 2023-01-10
BM-1150 | Priya Nair | Customer Success | Tom Becker | 2023-05-20


## 12. Compare Chunking Strategies Systematically

For every strategy built in this step, we record: number of chunks, average
chunk length in characters, and total embedding tokens (a proxy for indexing
cost — more tokens means a larger embedding bill). Retrieval recall and
reranker quality are approximated by the boundary-crossing results from
Section 9, since running full retrieval for every strategy would require
re-indexing each chunk set into Elasticsearch/Chroma.

In [27]:
def average_chunk_length_chars(chunks):
    if not chunks:
        return 0
    return round(sum(len(chunk["text"]) for chunk in chunks) / len(chunks), 1)


def total_tokens_for_chunks(chunks, encoding):
    return sum(
        chunk["token_count"] if "token_count" in chunk else len(encoding.encode(chunk["text"]))
        for chunk in chunks
    )


strategy_chunk_sets = {
    "fixed_character": char_chunks,
    "token_based": token_chunks,
    "paragraph_based": paragraph_chunks,
    "heading_aware (sample.md)": sample_md_heading_chunks,
    "recursive": recursive_chunks,
    "parent_child (children)": child_chunks,
    "parent_child (parents)": parent_chunks,
}

header = f"{'strategy':<28} {'num_chunks':>11} {'avg_chars':>10} {'total_tokens':>13}"
print(header)
print("-" * 65)

comparison_rows = []
for name, chunks in strategy_chunk_sets.items():
    row = {
        "strategy": name,
        "num_chunks": len(chunks),
        "avg_chunk_chars": average_chunk_length_chars(chunks),
        "total_embedding_tokens": total_tokens_for_chunks(chunks, encoding),
    }
    comparison_rows.append(row)
    line = f"{name:<28} {row['num_chunks']:>11} {row['avg_chunk_chars']:>10} {row['total_embedding_tokens']:>13}"
    print(line)

NameError: name 'char_chunks' is not defined

### Reflection

- **Chunks too small** — facts fragment across boundaries (Section 9's
  `begins_end_continues_next` case); more chunks means more embedding calls
  and a higher chance the reranker sees near-duplicate, low-context
  candidates.
- **Chunks too large** — fewer, cheaper chunks, but each one dilutes the
  embedding with unrelated content, hurting retrieval precision, and risks
  exceeding the generation context budget.
- **Overlap** — Section 8 shows overlap trades duplicated tokens (higher
  indexing cost, redundant candidates) for a better chance a boundary fact
  survives inside one chunk. It does not fix answers that genuinely span two
  sections (Section 9's `requires_two_sections` case) — only parent-child
  retrieval or multi-chunk generation contexts solve that.
- **Same strategy for everything?** — No. Narrative text (profile.txt) suits
  paragraph or recursive chunking; structured Markdown (sample.md) benefits
  from heading-aware chunking so retrieved chunks stay traceable to their
  section; tables need row-based chunking for point lookups, or full-table
  chunking when the query needs the whole table at once.
- **Headings in child chunks?** — Yes for retrieval (the heading gives the
  embedding topical context) and yes for generation (the parent should carry
  its heading path so the model knows what section it is reading).
- **Chunk size vs. context size** — chunk size controls what one embedding
  represents and what one retrieval candidate looks like; context size is the
  total text handed to the generation model, typically several chunks (or
  their resolved parents) concatenated together.
- **When is semantic chunking worth it?** — When section/paragraph boundaries
  don't reliably track topic changes (e.g. transcripts, chat logs, freeform
  narrative) and the extra embedding-based boundary detection pays for itself
  in retrieval quality.